# NB03B: CARD-BS EXTRACTION & CLIPPING

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


Extracts VV/VH bands from CARD-BS zip files, reprojects to WGS84, clips to bbox (or polygon), converts to dB.

Requires: NB02 DL-CARD completed (zips in CARD_ZIP_DIR), NB01 bbox_aligned.geojson per city.

Input: CARD zips + card_download_tracker.json
Output: SAR_CARD/{city}/{period}/{city}_CARD_{pol}_{date}.tif

# CELL 4: COMPLETE ENVIRONMENT SETUP (BDA + SNAP12) - LOCAL WSL VERSION


In [1]:
# @title CELL 3: NB03b CONFIG
# ---- shared ----
TIER_SELECTION = [0]
CITY_SELECTION = ["Mariupol"]              # or ["Mariupol"]
# ---- pre-audit (cell 7) ----
FIX_TRACKER = True
NAN_THRESHOLD = 0.90
MIN_FILE_BYTES = 1024
BBOX_TOLERANCE = 0.001
MIN_PIXEL_DIM = 10
RASTERIO_SAMPLE_PER_CITY = 5
# ---- polygon mask (cell 11) ----
POLYGON_MASK_DRY_RUN = True
# ---- audit (cell 14) ----
MIN_SLC_SIZE = 3e9
# ---- reconcile (cell 16) ----
FIX_MODE = True
BACKUP_BEFORE_FIX = True

DEM_BUFFER = 0.15
MIN_SLC_SIZE = 3e9
MIN_CARD_SIZE = 1e9
REQUIRE_SAR_ALIGNED = True
SHORT_CONFLICT_THRESHOLD_DAYS = 30
# @title CELL CONFIG: NB03e
CLIP_BY_POLYGON = False
DRY_RUN_PRUNE = False
PRUNE_SELECTOR = ['stale_date', 'wrong_extent', 'corrupt']  # or 'ALL' 'superseded_coh', 'stale_date', 'wrong_extent', 'surplus_city', 'corrupt'] 
VERBOSE = True

In [2]:
# @title CELL 4: LOAD GLOBAL SETUP
import platform, os
if platform.system() == 'Windows':
    _setup = r'F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py'
elif os.path.exists('/content/drive_f'):
    _setup = '/content/drive_f/masterthesis/notebooks/global_setup.py'
else:
    _setup = '/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py'
with open(_setup) as f:
    exec(f.read())


BDA GLOBAL SETUP
Started: 2026-04-06 02:02:38
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0]
  CITY_SELECTION: ['Mariupol']
  REQUIRE_UNOSAT: False
  CITIES_TO_PROCESS: 1 cities
    Mariupol (battle_start=2022-02-24)

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:501: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     910.8/7452.0 GB (6541.3 GB free)
  GDrive (F:)     794.9/3726.0 GB (2931.1 GB free)
  Local data      9941.6/14901.9 GB (4960.2 GB free)
  Data stack      794.9/3726.0 GB (2931.1 GB free)
  WSL ext4        110.4/1006.9 GB (845.3 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 1, CITY=Mariupol
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_bbo

# CELL 4B: DATA VERIFICATION

In [3]:
# @title CELL 4B: DATA VERIFICATION
import sys, importlib
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import data_verification
importlib.reload(data_verification)
from data_verification import run as run_data_verification

VERIFICATION_STATUS = run_data_verification(
    satellite_dirs=dict(
        SAR_COH_DIR=SAR_COH_DIR,
        SAR_CARD_DIR=SAR_CARD_DIR,
        MS_DIR=MS_DIR,
        MS_METADATA_DIR=MS_METADATA_DIR,
        SAR_METADATA_DIR=SAR_METADATA_DIR,
        LANDUSE_DIR=LANDUSE_DIR,
        TRANSITION_DIR=TRANSITION_DIR,
    ),
    reference_dirs=dict(
        CITIES_DIR=CITIES_DIR,
        UKR_BOUNDARIES=UKR_BOUNDARIES,
        OSM_2022=OSM_2022,
        OSM_2025=OSM_2025,
        OSM_OUTPUT_DIR=OSM_OUTPUT_DIR,
        UNOSAT_DIR=UNOSAT_DIR,
        UNOSAT_RAW_DIR=UNOSAT_RAW_DIR,
        XBD_ROOT=XBD_ROOT,
    ),
    ml_dirs=dict(
        CHECKPOINTS_DIR=CHECKPOINTS_DIR,
        PRETRAINED_DIR=PRETRAINED_DIR,
        TRAINED_DIR=TRAINED_DIR,
    ),
    result_dirs=dict(
        PREDICTIONS_DIR=PREDICTIONS_DIR,
        VALIDATION_DIR=VALIDATION_DIR,
        VISUALIZATIONS_DIR=VISUALIZATIONS_DIR,
        OUTPUTS_DIR=OUTPUTS_DIR,
        LOGS_DIR=LOGS_DIR,
    ),
    local_dirs=dict(
        RAW_SLC_ZIP=RAW_SLC_ZIP,
        CARD_ZIP_DIR=CARD_ZIP_DIR,
        MS_ZIP_DIR=MS_ZIP_DIR,
        SAR_SLC_ORBIT_DIR=SAR_SLC_ORBIT_DIR,
        DEM_DIR=DEM_DIR,
    ),
    snap_dirs=dict(
        SNAP_WORKDIR=SNAP_WORKDIR,
        TEMP_DOWNLOAD_DIR=TEMP_DOWNLOAD_DIR,
        CROSSBATTLE_STASH_DIR=CROSSBATTLE_STASH_DIR,
        SNAP_GRAPH_DIR=SNAP_GRAPH_DIR,
    ),
    tracker_files=dict(
        INSAR_TRACKER_FILE=INSAR_TRACKER_FILE,
        MS_TRACKING_FILE=MS_TRACKING_FILE,
        CARD_TRACKER_FILE=CARD_TRACKER_FILE,
        PROGRESS_FILE=PROGRESS_FILE,
        CITIES_PKL_FILE=CITIES_PKL_FILE,
        MS_CITIES_PKL_FILE=MS_CITIES_PKL_FILE,
    ),
    gpt_path=GPT_PATH,
    content_local=CONTENT_LOCAL,
    get_disk_space_fn=get_disk_space,
)

DATA VERIFICATION
  [   OK   ] SAR COH (coherence products)           850 files, 6.3 GB
  [   OK   ] SAR CARD-BS (calibrated)              3597 files, 20.1 GB
  [   OK   ] Multispectral (Sentinel-2)           62894 files, 119.8 GB
  [   OK   ] MS Metadata                             52 files, 2.1 MB
  [   OK   ] SAR Metadata                            52 files, 19.4 MB
  [   OK   ] Landuse Classification               21320 files, 116.0 GB
  [   OK   ] Transition Matrix                        1 files, 0.0 MB
  [   OK   ] City Boundaries                        425 files, 5.7 GB
  [   OK   ] Ukraine Admin Boundaries                60 files, 139.5 MB
  [   OK   ] OSM Feb 2022 (pre-conflict)             92 files, 3.7 GB
  [   OK   ] OSM Oct 2025 (current)                  92 files, 4.6 GB
  [   OK   ] OSM Buildings (extracted)              123 files, 848.6 MB
  [   OK   ] UNOSAT Damage Assessments             1048 files, 1.2 GB
  [   OK   ] xBD Dataset                          11203 files,

# MEDIA SCANNER

In [4]:
# @title CELL PRE-AUDIT: PRODUCT VALIDATOR (CARD)
import sys, importlib
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
import product_pre_audit
importlib.reload(product_pre_audit)
from product_pre_audit import run as run_pre_audit
DELETE_BAD = True
PRE_AUDIT = run_pre_audit(
    sensors=['card'],
    sar_card_dir=SAR_CARD_DIR,
    cities_dir=CITIES_DIR,
    cities_to_process=CITIES_TO_PROCESS,
    delete_bad=DELETE_BAD,
    rasterio_sample_per_city=3,
)

PRODUCT PRE-AUDIT
  Sensors: ['card']
  DELETE_BAD: True
  Sample per city: 3
  Cities filter: 1 cities

CARD-BS PRODUCTS (SAR_CARD_DIR)
  Dir: /content/drive_f/masterthesis/data/satellite/SAR_CARD
  Cities: 1, TIFs: 56

  City                       Files    OK   Bad Issues
  ----------------------------------------------------------------------
  Mariupol                      56    56     0 OK

  CARD totals: OK=56, tiny=0, NaN=0, bbox=0, dim=0, corrupt=0, deleted=0

PRE-AUDIT COMPLETE (6s)
  CARD : OK=56, tiny=0, NaN=0, bbox=0, dim=0, corrupt=0, deleted=0
  Total files deleted: 0


# CELL 14A3: LOAD SCENES

In [5]:
# @title CELL 14A3: LOAD SCENES
FORCE_RERUN = False

import sys, importlib
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import scene_loader
importlib.reload(scene_loader)
from scene_loader import run as run_scene_loader

cities_df = run_scene_loader(
    cities_dir=CITIES_DIR,
    sar_metadata_dir=SAR_METADATA_DIR,
    cities_pkl_file=CITIES_PKL_FILE,
    force_rerun=FORCE_RERUN,
)

CELL 14A3: LOAD SCENES FROM GOOGLE DRIVE

  Loading cities dataframe from pickle...
  Source: /content/drive_f/masterthesis/data/satellite/SAR_METADATA/cities_dataframe.pkl
  Loaded 50 cities from cache

CITIES DATAFRAME LOADED
Source: /content/drive_f/masterthesis/data/satellite/SAR_METADATA
Cache: /content/drive_f/masterthesis/data/satellite/SAR_METADATA/cities_dataframe.pkl

Cities summary:
   1. [T0] Avdiivka            : 63 pre, 93 post, 60 battle, Battle: 2022-02-24 - 2024-02-17 | Orbit: 94
   2. [T0] Bakhmut             : 79 pre, 115 post, 31 battle, Battle: 2022-05-17 - 2023-05-20 | Orbit: 43
   3. [T0] Mariupol            : 63 pre, 146 post, 7 battle, Battle: 2022-02-24 - 2022-05-20 | Orbit: 43
   4. [T0] Sievierodonetsk     : 59 pre, 144 post, 5 battle, Battle: 2022-05-05 - 2022-06-25 | Orbit: 145
   5. [T1] Chasiv Yar          : 34 pre, 16 post, 117 battle, Battle: 2023-06-01 - ongoing | Orbit: 145
   6. [T1] Hirske              : 59 pre, 141 post, 3 battle, Battle: 2022-06-

# CELL 14A3-MS: LOAD MULTISPECTRAL SCENES

In [6]:
# @title CELL 14A3-MS: LOAD MULTISPECTRAL SCENES
FORCE_RERUN = False

import sys, importlib
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import ms_scene_loader
importlib.reload(ms_scene_loader)
from ms_scene_loader import run as run_ms_scene_loader

ms_cities_df = run_ms_scene_loader(
    cities_dir=CITIES_DIR,
    ms_metadata_dir=MS_METADATA_DIR,
    ms_cities_pkl_file=MS_CITIES_PKL_FILE,
    force_rerun=FORCE_RERUN,
)

CELL 14A3-MS: LOAD MULTISPECTRAL SCENES FROM GOOGLE DRIVE

  Loading MS cities dataframe from pickle...
  Source: /content/drive_f/masterthesis/data/satellite/MS/metadata/ms_cities_dataframe.pkl
  Loaded 50 cities from cache

MS CITIES DATAFRAME LOADED
Source: /content/drive_f/masterthesis/data/satellite/MS/metadata
Cache: /content/drive_f/masterthesis/data/satellite/MS/metadata/ms_cities_dataframe.pkl

Tier summary:
  Tier 0: 4 cities, 86 total scenes
  Tier 1: 15 cities, 480 total scenes
  Tier 2: 24 cities, 662 total scenes
  Tier 3: 7 cities, 482 total scenes

Cities missing pre-battle scenes (2):
    [T2] Borodyanka
    [T2] Mykolaiv

Cities missing post-battle scenes (6):
    [T2] Borodyanka
    [T2] Mykolaiv
    [T2] Vovchansk
    [T3] Dnipro
    [T3] Kryvyi Rih
    [T3] Zaporizhzhia

Cities detail:
   1. [T0] Avdiivka             | 2 pre, 2 post, 18 battle, 3 preBL, 5 postBL | SAR | 2022-02-24 - 2024-02-17
   2. [T0] Bakhmut              | 2 pre, 2 post, 8 battle, 5 preBL, 5 po

# CELL DL-CARD-EXTRACT: EXTRACT & CLIP CARD ZIPS TO CITY TIFS

In [7]:
# @title CELL DL-CARD-EXTRACT: EXTRACT & CLIP CARD ZIPS TO CITY TIFS
# =============================================================================
# Reads card_download_tracker.json for zip_ready entries, extracts VV/VH bands,
# reprojects to WGS84, clips to bbox (default) or polygon, converts to dB.
#
# Run AFTER: DL-CARD (download-only mode, EXTRACT_AFTER_DOWNLOAD=False)
# Always clips to aoi_bbox from AOI.geojson (flat output, no period subdirs)
#
# Output: SAR_CARD/{city}/s1__{pol}__{date}.tif
# =============================================================================

import json
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask as rasterio_mask
from rasterio.io import MemoryFile
from rasterio.transform import from_bounds as transform_from_bounds
from pathlib import Path
from datetime import datetime
import geopandas as gpd
from shapely.geometry import mapping
import zipfile
import shutil
import time

FORCE_RERUN = False
DRY_RUN = False

CELL_ID = "cell_dl_card_extract"
# City filter: uses CITIES_TO_PROCESS from Cell 4 (resolved from CITY_SELECTION + TIER_SELECTION)

print("=" * 80)
print("CELL DL-CARD-EXTRACT: EXTRACT & CLIP CARD ZIPS")
print("=" * 80)
print(f"  Timestamp: {datetime.now().isoformat()}")

def _card_output_root():
    return SAR_CARD_DIR
print(f"  FORCE_RERUN: {FORCE_RERUN}")

# =============================================================================
# LOAD TRACKER
# =============================================================================

if not CARD_TRACKER_FILE.exists():
    raise ValueError(f"Card tracker not found: {CARD_TRACKER_FILE}. Run DL-CARD first.")

with open(CARD_TRACKER_FILE) as f:
    card_tracker = json.load(f)

def save_card_tracker():
    with open(CARD_TRACKER_FILE, 'w') as f:
        json.dump(card_tracker, f, indent=2)
        
# Clean stale tracker entries (zip pruned but tracker not updated)
stale_count = 0
for key, entry in list(card_tracker.items()):
    zp = entry.get('zip_path', '')
    if zp and not Path(zp).exists():
        # check if any zip for this date exists via pattern search
        date_str = entry.get('date', '')
        if date_str:
            date_compact = date_str.replace('-', '')
            pattern = f"*_{date_compact}T*_CARD_BS.zip"
            if not list(CARD_ZIP_DIR.glob(pattern)):
                entry['status'] = 'zip_pruned'
                stale_count += 1
if stale_count:
    save_card_tracker()
    print(f"  Marked {stale_count} stale entries as zip_pruned")
    
# find entries needing extraction
# disk-is-truth: queue ALL downloaded entries, let disk check at line 334 skip existing
extract_tasks = []
for key, entry in card_tracker.items():
    status = entry.get('status', '')
    if status in ('success', 'exists', 'zip_ready'):
        extract_tasks.append((key, entry))

# Filter by CITIES_TO_PROCESS (resolved from CITY_SELECTION + TIER_SELECTION)
if CITIES_TO_PROCESS:
    filtered = []
    for key, entry in extract_tasks:
        parts = key.rsplit('_', 3)
        if len(parts) >= 4:
            city = '_'.join(parts[:-3])
        else:
            city = entry.get('city', key.split('_')[0])
        if city in CITIES_TO_PROCESS:
            filtered.append((key, entry))
    extract_tasks = filtered

print(f"  Tracker entries: {len(card_tracker)}")
print(f"  CITIES_TO_PROCESS: {len(CITIES_TO_PROCESS)} cities")
print(f"  Tasks to extract: {len(extract_tasks)}")

# =============================================================================
# LOAD CLIP GEOMETRIES PER CITY
# =============================================================================

clip_geoms = {}  # city -> geometry (bbox or polygon)

def _load_aoi_feature(city_name, feature_type):
    """Fast AOI.geojson loader — json.load + scan, no geopandas (avoids 131MB parse)."""
    import json as _j
    from shapely.geometry import shape as _s
    aoi_file = CITIES_DIR / city_name / "AOI.geojson"
    if not aoi_file.exists():
        return None
    with open(aoi_file) as f:
        gj = _j.load(f)
    for feat in gj['features']:
        if feat.get('properties', {}).get('feature_type') == feature_type:
            return _s(feat['geometry'])
    return None

def get_clip_geom(city_name):
    if city_name in clip_geoms:
        return clip_geoms[city_name]

    try:
        geom = load_aoi_bbox(city_name, CITIES_DIR)
        clip_geoms[city_name] = geom
        return geom
    except (FileNotFoundError, ValueError):
        print(f"      ERROR: no aoi_bbox in AOI.geojson for {city_name}")
        return None

# =============================================================================
# ZIP LOOKUP: find ALL candidate zips on disk for a date
# =============================================================================

def find_card_zips(entry, date_str):
    """Find ALL CARD zips on disk for a date. Returns list of Paths.
    Multiple zips per date = different swath segments. Only one may cover the city."""
    results = []

    # 1) tracker zip_path
    zp = entry.get('zip_path', '')
    if zp and Path(zp).exists():
        results.append(Path(zp))

    # 2) scene_name in CARD_ZIP_DIR
    sn = entry.get('scene_name', '')
    if sn:
        sn_base = sn.replace('.zip', '')
        candidate = CARD_ZIP_DIR / f"{sn_base}.zip"
        if candidate.exists() and candidate not in results:
            results.append(candidate)

    # 3) date pattern search in CARD_ZIP_DIR - ALL matches
    date_compact = date_str.replace('-', '')
    pattern = f"*_{date_compact}T*_CARD_BS.zip"
    for match in sorted(CARD_ZIP_DIR.glob(pattern)):
        if match not in results:
            results.append(match)

    return results

# =============================================================================
# EXTRACT FUNCTION (from DL-CARD, unchanged except clip geometry source)
# =============================================================================

def extract_and_clip(zip_path, city_name, clip_geom, date_str, period, output_dir):
    zip_path = Path(zip_path)
    if not zip_path.exists():
        print(f"      ZIP NOT FOUND")
        return []
    if not zipfile.is_zipfile(zip_path):
        size_mb = zip_path.stat().st_size / (1024**2)
        print(f"      BAD ZIP ({size_mb:.1f} MB) - DELETING: {zip_path.name}")
        zip_path.unlink()
        return []

    extract_dir = CARD_TEMP_DIR / 'extract'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)

    raster_files = []
    for ext in ['*.img', '*.tiff', '*.tif']:
        raster_files.extend(extract_dir.rglob(ext))

    bands = {}
    for rf in raster_files:
        fname = rf.stem.lower()
        if 'vv' in fname:
            bands['VV'] = rf
        elif 'vh' in fname:
            bands['VH'] = rf

    if 'VV' not in bands or 'VH' not in bands:
        print(f"      WARNING: Missing VV/VH bands")
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        return []

    output_dir.mkdir(parents=True, exist_ok=True)
    saved_files = []

    for pol, raster_path in bands.items():
        out_name = f"s1__{pol.lower()}__{date_str.replace('-','')}.tif"
        out_path = output_dir / out_name

        with rasterio.open(raster_path) as src:
            dst_crs = rasterio.crs.CRS.from_epsg(4326)

            clip_bounds = clip_geom.bounds
            buf = 0.01
            dst_bounds = (clip_bounds[0]-buf, clip_bounds[1]-buf,
                          clip_bounds[2]+buf, clip_bounds[3]+buf)

            dst_transform_full, _, _ = calculate_default_transform(
                src.crs, dst_crs, src.width, src.height, *src.bounds,
            )
            res = abs(dst_transform_full.a)

            dst_width_clip = max(1, int((dst_bounds[2]-dst_bounds[0])/res))
            dst_height_clip = max(1, int((dst_bounds[3]-dst_bounds[1])/res))
            dst_transform_clip = transform_from_bounds(*dst_bounds, dst_width_clip, dst_height_clip)

            dst_data = np.empty((1, dst_height_clip, dst_width_clip), dtype=np.float32)

            reproject(
                source=rasterio.band(src, 1),
                destination=dst_data[0],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=dst_transform_clip,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear,
            )

        dst_data_db = 10 * np.log10(np.maximum(dst_data, 1e-10))

        meta = {
            'driver': 'GTiff', 'dtype': 'float32',
            'width': dst_width_clip, 'height': dst_height_clip,
            'count': 1, 'crs': dst_crs, 'transform': dst_transform_clip,
            'nodata': np.nan,
        }

        with MemoryFile() as memfile:
            with memfile.open(**meta) as mem_dst:
                mem_dst.write(dst_data_db)
            with memfile.open() as mem_src:
                try:
                    clipped, clipped_transform = rasterio_mask(
                        mem_src, [mapping(clip_geom)], crop=True, nodata=np.nan
                    )
                except Exception as e:
                    print(f"      Clip failed for {pol}: {e}")
                    continue

        out_meta = {
            'driver': 'GTiff', 'dtype': 'float32',
            'width': clipped.shape[2], 'height': clipped.shape[1],
            'count': 1, 'crs': dst_crs, 'transform': clipped_transform,
            'nodata': np.nan, 'compress': 'lzw',
        }

        with rasterio.open(out_path, 'w', **out_meta) as dst:
            dst.write(clipped)

        valid = clipped[0][(~np.isnan(clipped[0])) & (clipped[0] > -100)]
        if len(valid) > 0:
            print(f"      {pol}: {out_name}  {clipped.shape[1:]}  "
                  f"mean={np.mean(valid):.1f}dB  [{np.min(valid):.1f}, {np.max(valid):.1f}]")
        else:
            print(f"      {pol}: {out_name}  NO VALID DATA")

        saved_files.append(out_path)

    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    return saved_files


def check_tifs_have_valid_data(file_list):
    """Check if any of the output TIFs contain valid (non-NaN, > -100dB) data."""
    for f in file_list:
        f = Path(f)
        if not f.exists():
            continue
        with rasterio.open(f) as src:
            data = src.read(1)
            valid = data[(~np.isnan(data)) & (data > -100)]
            if len(valid) > 0:
                return True
    return False

# =============================================================================
# PROCESS
# =============================================================================

if DRY_RUN:
    print(f"\n  DRY RUN - would extract {len(extract_tasks)} tasks")
    for key, entry in extract_tasks[:20]:
        print(f"    {key}: {entry.get('zip_path', '?')}")
    if len(extract_tasks) > 20:
        print(f"    ... and {len(extract_tasks)-20} more")

elif extract_tasks:
    total_start = time.time()
    success_count = 0
    skip_count = 0
    fail_count = 0
    segment_fallback_count = 0

    for ti, (tracker_key, entry) in enumerate(extract_tasks):
        city = tracker_key.split('_')[0]
        # handle city names with underscores: everything before last _YYYY-MM-DD
        parts = tracker_key.rsplit('_', 3)
        if len(parts) >= 4:
            city = '_'.join(parts[:-3])
            date_str = f"{parts[-3]}-{parts[-2]}-{parts[-1]}"
        else:
            city = entry.get('city', tracker_key.split('_')[0])
            date_str = entry.get('date', '')

        # use entry fields directly
        date_str = entry.get('date', date_str)
        period = entry.get('period', 'unknown')
        card_root = _card_output_root()
        output_dir = card_root / city
        expected_vv = output_dir / f"s1__vv__{date_str.replace('-','')}.tif"
        expected_vh = output_dir / f"s1__vh__{date_str.replace('-','')}.tif"
        # fallback: old naming
        if not expected_vv.exists():
            expected_vv = output_dir / f"{city}_CARD_VV_{date_str.replace('-','')}.tif"
        if not expected_vh.exists():
            expected_vh = output_dir / f"{city}_CARD_VH_{date_str.replace('-','')}.tif"

        # skip if already extracted (unless overwrite mode)
        if not FORCE_RERUN and expected_vv.exists() and expected_vh.exists():
            skip_count += 1
            continue

        print(f"\n  [{ti+1}/{len(extract_tasks)}] {city} / {period} / {date_str}")

        clip_geom = get_clip_geom(city)
        if clip_geom is None:
            print(f"      SKIP: no boundary/bbox for {city}")
            fail_count += 1
            continue

        zip_candidates = find_card_zips(entry, date_str)
        if not zip_candidates:
            print(f"      SKIP: zip not found (scene={entry.get('scene_name','?')}, date={date_str})")
            fail_count += 1
            continue

        # delete existing TIFs if FORCE_RERUN (clean re-extract)
        if FORCE_RERUN:
            for old_tif in [expected_vv, expected_vh]:
                if old_tif.exists():
                    old_tif.unlink()

        # try each candidate zip until one produces valid data
        got_valid = False
        saved = []

        for zi, zip_path in enumerate(zip_candidates):
            zip_path_str = str(zip_path)
            try:
                saved = extract_and_clip(zip_path_str, city, clip_geom, date_str, period, output_dir)
                if saved and check_tifs_have_valid_data(saved):
                    got_valid = True
                    if zi > 0:
                        segment_fallback_count += 1
                    break
                elif saved and len(zip_candidates) > 1 and zi < len(zip_candidates) - 1:
                    print(f"      -> no valid data from segment {zi+1}/{len(zip_candidates)}, trying next...")
            except Exception as e:
                if zi < len(zip_candidates) - 1:
                    print(f"      -> segment {zi+1} failed: {e}, trying next...")
                else:
                    print(f"      Extract failed: {e}")
                    import traceback
                    traceback.print_exc()

        if got_valid:
            card_tracker[tracker_key]['status'] = 'success'
            card_tracker[tracker_key]['files'] = [str(f.relative_to(card_root)) for f in saved]
            card_tracker[tracker_key]['clip_mode'] = 'bbox'
            save_card_tracker()
            success_count += 1
        else:
            card_tracker[tracker_key]['status'] = 'no_valid_data'
            card_tracker[tracker_key]['segments_tried'] = len(zip_candidates)
            save_card_tracker()
            fail_count += 1

    total_elapsed = time.time() - total_start

    print(f"\n{'='*80}")
    print(f"CARD-BS EXTRACTION COMPLETE ({total_elapsed/60:.1f}min)")
    print(f"{'='*80}")
    print(f"  Success:  {success_count}")
    print(f"  Skipped:  {skip_count} (TIFs exist)")
    print(f"  Failed:   {fail_count}")
    print(f"  Segment fallbacks: {segment_fallback_count}")
    print(f"  Clip mode: bbox (always)")
    print(f"  Output root: {SAR_CARD_DIR}")

else:
    print("\n  Nothing to extract - all tasks already processed!")

print(f"\n{'='*80}")
print("DL-CARD-EXTRACT COMPLETE")
print(f"{'='*80}")


CELL DL-CARD-EXTRACT: EXTRACT & CLIP CARD ZIPS
  Timestamp: 2026-04-06T02:14:57.706330
  FORCE_RERUN: False
  Tracker entries: 2809
  CITIES_TO_PROCESS: 1 cities
  Tasks to extract: 28

CARD-BS EXTRACTION COMPLETE (0.0min)
  Success:  0
  Skipped:  28 (TIFs exist)
  Failed:   0
  Segment fallbacks: 0
  Clip mode: bbox (always)
  Output root: /content/drive_f/masterthesis/data/satellite/SAR_CARD

DL-CARD-EXTRACT COMPLETE


# CELL DL-CARD-BONUS: CLIP ALL CARD ZIPS TO ALL OVERLAPPING CITIES

# BONUS CITIES REMOVED (write in thesis, discovered while doing QA)
- clipping of bonus cities which do not have the same orbit selected as run city corrupts ML pipeline!

# CELL POLYGON-MASK: MASK SAR PRODUCTS TO CITY POLYGON

In [8]:
# @title CELL POLYGON-MASK: MASK PRODUCTS TO CITY POLYGON
POLYGON_MASK_ENABLED = False
FORCE_RERUN = False
SKIP_EXISTING = True
DRY_RUN = False

import sys, importlib
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import polygon_mask
importlib.reload(polygon_mask)
from polygon_mask import run as run_polygon_mask

run_polygon_mask(
    modalities=dict(
        COH={'src': SAR_COH_DIR, 'dst': SAR_COH_CITY_POLYGON},
        CARD={'src': SAR_CARD_DIR, 'dst': SAR_CARD_CITY_POLYGON},
        MS={'src': MS_DIR, 'dst': MS_CITY_POLYGON},
    ),
    cities_dir=CITIES_DIR,
    city_selection=CITY_SELECTION,
    enabled=POLYGON_MASK_ENABLED,
    mask_coh=True,
    mask_card=True,
    mask_ms=False,
    force_rerun=FORCE_RERUN,
    skip_existing=SKIP_EXISTING,
    dry_run=DRY_RUN,
)

POLYGON MASK: SKIPPED (enabled=False)


{}

# CELL 14D-VERIFY: PRODUCT COMPLETENESS

In [9]:
# @title CELL 14D-VERIFY: PRODUCT COMPLETENESS
import sys, importlib
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
import product_verify
importlib.reload(product_verify)
from product_verify import run as run_product_verify

try:
    VERIFICATION_RESULTS, VERIFICATION_SUMMARY = run_product_verify(
        sar_coh_dir=SAR_COH_DIR,
        sar_card_dir=SAR_CARD_DIR,
        ms_dir=MS_DIR,
        cities_dir=CITIES_DIR,
        insar_tracker_file=INSAR_TRACKER_FILE,
        tier_selection=TIER_SELECTION,
        city_selection=CITY_SELECTION,
        cities_to_process=CITIES_TO_PROCESS if 'CITIES_TO_PROCESS' in dir() else None,
    )
except (json.JSONDecodeError, Exception) as e:
    print(f'  VERIFY FAILED: {type(e).__name__}: {e}')
    print(f'  INSAR_TRACKER_FILE: {INSAR_TRACKER_FILE}')
    _sz = INSAR_TRACKER_FILE.stat().st_size if INSAR_TRACKER_FILE.exists() else -1
    print(f'  File size: {_sz} bytes {"(EMPTY!)" if _sz == 0 else ""}')
    print(f'  Fix: delete or regenerate tracker via NB03a, then rerun.')
    VERIFICATION_RESULTS = None
    VERIFICATION_SUMMARY = None

CELL 14D-VERIFY: PRODUCT COMPLETENESS VERIFICATION
Timestamp: 2026-04-06T02:15:00.422292

Verification scope:
  TIER_SELECTION:  [0]
  CITY_SELECTION:  ['Mariupol']
  SAR_COH_DIR:     /content/drive_f/masterthesis/data/satellite/SAR_COH
  SAR_CARD_DIR:    /content/drive_f/masterthesis/data/satellite/SAR_CARD
  MS_DIR:          /content/drive_f/masterthesis/data/satellite/MS

Cities to verify: 1
  InSAR tracker loaded: 21 cities

COH (COHERENCE) PRODUCTS — flat: SAR_COH/{city}/
City                       BL    PRE  CROSS   POST  Other  Total   Status
                        pairs  pairs  pairs  pairs   tifs   tifs
------------------------------------------------------------------------------------------
  Mariupol                  0      0      0      0      0      4    EMPTY

  COH Summary: 0 OK, 0 partial, 1 missing/empty

CARD-BS PRODUCTS — flat: SAR_CARD/{city}/
City                      Dates     VV     VH    Total   TempSt   Status
-------------------------------------------------

In [10]:
# @title CELL SQLITE: REGISTER CARD PRODUCTS
import sys, importlib
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import product_catalog
importlib.reload(product_catalog)
from product_catalog import ensure_schema, register_card_products

print(f"CATALOG_DB: {CATALOG_DB}")
print(f"  exists: {CATALOG_DB.exists()}")

ensure_schema(CATALOG_DB)

import sqlite3
_conn = sqlite3.connect(str(CATALOG_DB))
_tables = [r[0] for r in _conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
_conn.close()
print(f"  tables after ensure_schema: {_tables}")

register_card_products(CATALOG_DB, SAR_CARD_DIR, cities=CITIES_TO_PROCESS)

# verify
_conn = sqlite3.connect(str(CATALOG_DB))
_cur = _conn.execute("SELECT modality, COUNT(*), ROUND(SUM(size_mb),0) FROM product_inventory GROUP BY modality")
rows = _cur.fetchall()
_conn.close()
print(f"\n  product_inventory contents:")
for mod, cnt, mb in rows:
    print(f"    {mod:8s}: {cnt:6d} rows ({mb:.0f} MB)")


CATALOG_DB: /mnt/f/PROJECTS/masterthesis/data_stack/bda.sqlite
  exists: True
  product_catalog: schema ready -> /mnt/f/PROJECTS/masterthesis/data_stack/bda.sqlite
  tables after ensure_schema: ['cities', 'scenes', 'products', 'processing_runs', 'sqlite_sequence', 'data_stack', 'features', 'rename_lut', 'product_inventory', 'product_audit_log', 'zip_inventory', 'zip_audit_log', 'date_prune_log', 'source_product_scan', 'derived_product_scan', 'stack_audit', 'qa_results', 'qa_run_summary']
  product_catalog: registered 56 CARD products for Mariupol
  product_catalog: CARD total = 56 products across 1 cities

  product_inventory contents:
    CARD    :  42070 rows (214017 MB)
    COH     :   5873 rows (33700 MB)
    MS      :  43278 rows (53571 MB)
    landuse :    285 rows (89121 MB)


# CELL AUDIT: SCAN ALL NB03 DERIVED PRODUCTS

In [11]:
# @title CELL AUDIT: SCAN ALL NB03A-D DERIVED PRODUCTS
import sys, importlib
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
import product_audit
importlib.reload(product_audit)
from product_audit import run as run_product_audit

NB04_AUDIT = run_product_audit(
    sar_coh_dir=SAR_COH_DIR,
    sar_card_dir=SAR_CARD_DIR,
    ms_dir=MS_DIR,
    landuse_dir=LANDUSE_DIR,
    insar_tracker_file=INSAR_TRACKER_FILE,
    card_tracker_file=CARD_TRACKER_FILE,
    outputs_dir=OUTPUTS_DIR,
    cities_to_process=CITIES_TO_PROCESS,
)

CELL AUDIT: SCAN ALL NB03A-D DERIVED PRODUCTS
  SAR_COH_DIR: /content/drive_f/masterthesis/data/satellite/SAR_COH
  SAR_CARD_DIR: /content/drive_f/masterthesis/data/satellite/SAR_CARD
  MS_DIR: /content/drive_f/masterthesis/data/satellite/MS
  LANDUSE_DIR: /content/drive_f/masterthesis/data/satellite/landuse_classification
  CITIES_TO_PROCESS: 1 cities

1. COHERENCE PRODUCTS (NB03A)

  Cities with COH products: 1
  Total TIFs: 4
  Total size: 114 MB (0.1 GB)
  Types: {'coherence': 4}

  City                        Pre  Post Cross    BL
  --------------------------------------------------
  Mariupol                      0     0     0     0

  Superseded COH pairs: 0 (clean)

  Tracker cross-check:
    Tracker cities: 21
    Disk cities:    1
    Tracked not on disk: {'Popasna', 'Bakhmut', 'New York', 'Soledar', 'Hirske', 'Sievierodonetsk', 'Pisky', 'Lysychansk', 'Volnovakha', 'Zolote', 'Krasnohorivka', 'Rubizhne', 'Maryinka', 'Toretsk', 'Vuhledar', 'Kramatorsk', 'Chasiv Yar', 'Avdiivka'

# CELL PRUNE: IDENTIFY AND REMOVE STALE/ORPHAN PRODUCTS

In [12]:
# @title CELL PRUNE: DERIVED PRODUCT PRUNE
# =============================================================================
# Uses NB04_AUDIT from cell above (no re-scan).
# Categories: superseded_coh, stale_date, wrong_extent, surplus_city, corrupt, empty_dir
# =============================================================================

import product_prune
importlib.reload(product_prune)
from product_prune import run as run_product_prune
DRY_RUN_PRUNE = False

PRUNE_RESULTS = run_product_prune(
    coh_raw=NB04_AUDIT['coh_raw'],
    card_raw=NB04_AUDIT['card_raw'],
    ms_raw=NB04_AUDIT['ms_raw'],
    coh_superseded=NB04_AUDIT.get('coh_superseded', {}),
    cities_dir=CITIES_DIR,
    valid_cities=CITIES_TO_PROCESS,
    product_dirs=[SAR_COH_DIR, SAR_CARD_DIR, MS_DIR, LANDUSE_DIR],
    dry_run=DRY_RUN_PRUNE,
    verbose=VERBOSE,
    prune_selector=PRUNE_SELECTOR,
    stale_date_buffer_days=60,
)

PRODUCT PRUNE: IDENTIFY AND REMOVE STALE/ORPHAN DERIVED PRODUCTS
  DRY_RUN: False
  PRUNE_SELECTOR: ['stale_date', 'wrong_extent', 'corrupt']
  STALE_DATE_BUFFER: 60 days
  Valid cities: 1
  Cities with temporal windows: 1

--- 1. Surplus city check ---
  Surplus city files: 0

--- 2. Superseded COH pairs ---
  Superseded COH pairs: 0

--- 3. Stale date check (buffer=60d) ---
  Stale date files: 0

--- 4. Wrong extent check ---
  Wrong extent: 0
  Corrupt: 0

--- 5. Empty directory check ---
  Empty directories: 3

PRUNE CANDIDATE SUMMARY
  Total candidates: 0
  Total size: 0 MB (0.0 GB)
    empty_dir                     :     3 dirs

PRUNE SELECTOR: ['corrupt', 'stale_date', 'wrong_extent']
  To delete: 0 files (0 MB)
  Kept:      0 files (0 MB)

PRUNE COMPLETE
  Deleted: 0 files (0 MB, 0.0 GB)
